# Test runs (Kaggle, T4 x2): every system on the frozen test set

Runs the baselines, the full system and its ablations on the private test split with the
frozen code (`docs/DECISIONS.md` sections 13 and 15). The 7B (AWQ) serves on GPU 0; the
verifier serves on GPU 1: the fine-tuned 3B, and at the end the untuned 3B for A4a.

**Notebook settings:** Accelerator *GPU T4 x2*, Internet *on*, about 4 to 5 hours.
**Inputs** (*Add Input*):
1. the private test dataset (the zip of `data/private/`; the notebook finds the folder
   that contains `cases/test/`);
2. the output of the `verifier_ft` notebook version that trained the verifier
   (`verifier_model/`, the merged model; if only the adapter is there, it is merged here).

If the GitHub repo is private, add a Kaggle secret `GITHUB_TOKEN` (read-only).

**Outputs, two zips:**
- `test_metrics.zip`: per run `config.yaml` and `metrics.json` (aggregate numbers only),
  the statistics, the manifest check, commit and `pip freeze`. Safe to share and commit.
- `test_private.zip`: per-case events, predictions, proposals and server logs. They
  contain test appeals: keep them private; never commit them or open them in Claude Code.

In [ ]:
REPO_URL = "https://github.com/JagadishReddyK7/regrade-appeals-agent.git"
BRANCH = "main"
VLLM_SPEC = "vllm==0.32.0"  # as in the dev runs
WORKDIR = "/kaggle/temp/regrade-appeals-agent"
OUT = "/kaggle/working/test_metrics"  # aggregate numbers only
PRIVATE = "/kaggle/working/test_private"  # per-case logs: test content
SERVED_PATH = "/kaggle/temp/verifier-ft"  # configs/pipeline_test.yaml serves this path
MERGE_DIR = "/kaggle/temp/verifier_model"  # used only if the input has just the adapter
# In priority order: if the session ends early, the most important runs are done.
FT_RUNS = [  # need the 7B and the fine-tuned 3B
    "pipeline_test",  # full system (main)
    "agent_test",  # B3: agent v0, whole appeal
    "b1_test",  # B1: direct decision
    "pipeline_no_claim_splitting_test",  # A0
    "pipeline_prompted_verifier_test",  # A4b (verifier on the 7B server)
    "pipeline_no_code_execution_test",  # A1
    "pipeline_no_retrieval_test",  # A2
    "pipeline_no_escalation_test",  # A6
]
NO_GPU_RUNS = ["b0_reject_test", "b0_accept_test"]
BASE_RUNS = ["pipeline_base_verifier_test"]  # A4a: untuned 3B on GPU 1, run last

In [ ]:
# Clone. The token (if any) is removed from the git remote afterwards.
import glob
import hashlib
import json
import os
import shutil
import subprocess
from pathlib import Path

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo, or no secret configured
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(WORKDIR)
os.makedirs(OUT, exist_ok=True)
os.makedirs(PRIVATE, exist_ok=True)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # same GPU numbering as nvidia-smi
!git log -1 --oneline

## Inputs: test set and fine-tuned verifier
Only counts and hashes are printed, never test content.

In [ ]:
# The test set: the input folder that contains cases/test, copied to data/private
# (the path the configs use; a copy, because the sandbox runs student code from there).
roots = sorted({str(Path(p).parent.parent) for p in
                glob.glob("/kaggle/input/**/cases/test", recursive=True)})
assert len(roots) == 1, f"expected one test dataset with cases/test, found {len(roots)}"
if os.path.exists("data/private"):
    shutil.rmtree("data/private")
shutil.copytree(roots[0], "data/private")
n_cases = len(glob.glob("data/private/cases/test/**/*.json", recursive=True))


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


# The frozen manifest (docs/TEST_SET_GUIDE.md step 5): "<sha256> data/private/<path>".
manifest = glob.glob("data/private/**/manifest.txt", recursive=True)
check = {"cases": n_cases, "manifest_found": bool(manifest)}
if manifest:
    listed = [line.split(maxsplit=1) for line in Path(manifest[0]).read_text().splitlines()
              if line.strip()]
    listed = [(d, p) for d, p in listed if not p.endswith("manifest.txt")]
    present = [(digest, path) for digest, path in listed if os.path.isfile(path)]
    check.update(
        manifest_sha256=sha256(manifest[0]),
        files_listed=len(listed),
        files_missing=len(listed) - len(present),
        files_changed=sum(sha256(path) != digest for digest, path in present),
    )
    check["frozen_ok"] = check["files_missing"] == 0 and check["files_changed"] == 0
json.dump(check, open(f"{OUT}/test_set_check.json", "w"), indent=2)
print(check)
if not check.get("frozen_ok"):
    print("WARNING: the test set does not match its manifest; check before trusting the runs")

In [ ]:
# The fine-tuned verifier: the merged model from the verifier_ft output, else its adapter.
merged = sorted(glob.glob("/kaggle/input/**/verifier_model/config.json", recursive=True))
adapters = sorted(glob.glob("/kaggle/input/**/runs/verifier_train-*/adapter/adapter_config.json",
                            recursive=True))
NEED_MERGE = not merged
if merged:
    model_dir = os.path.dirname(merged[-1])
    print("using the merged verifier from", model_dir)
else:
    assert adapters, "no verifier_model/ or adapter in the inputs: add the verifier_ft output"
    print("only the adapter found: it is merged after the install, from", adapters[-1])

In [ ]:
# Install. A merge (only if needed) runs before vLLM, with the training libraries, as in
# notebooks/verifier_ft.ipynb (torchao uninstalled: peft refuses the merge next to it).
def run(*args):
    done = subprocess.run(["python", "-m", *args], capture_output=True, text=True)
    print(done.stdout[-4000:], done.stderr[-4000:])
    return done.returncode


if NEED_MERGE:
    subprocess.run(["pip", "install", "-q", "-e", ".[gpu]"], check=True)
    subprocess.run(["pip", "uninstall", "-y", "-q", "torchao"], check=True)
    source = os.path.dirname(os.path.dirname(adapters[-1]))
    target = f"runs/{os.path.basename(source)}"
    shutil.copytree(source, target, dirs_exist_ok=True)
    assert run("verifier.train", "--config", "configs/verifier_train.yaml",
               "--merge-only", target) == 0, "merge failed: see the output above"
    if os.path.exists(MERGE_DIR):
        shutil.rmtree(MERGE_DIR)
    shutil.move(f"{target}/merged", MERGE_DIR)
    shutil.rmtree(target)  # not part of the test runs
    model_dir = MERGE_DIR
if os.path.lexists(SERVED_PATH):
    os.remove(SERVED_PATH)
os.symlink(model_dir, SERVED_PATH)
!pip install -q -e .
!pip install -q "{VLLM_SPEC}"
!pip uninstall -y -q torchaudio
!python -c "import torch, vllm; print('torch', torch.__version__, 'vllm', vllm.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Servers and helpers

In [ ]:
import signal
import time


def gpu_used_mb(gpu):
    query = ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits",
             "-i", str(gpu)]
    return int(subprocess.run(query, capture_output=True, text=True).stdout.strip() or 0)


def stop_server(stem, gpu):
    # Stop a vLLM server and its workers (one process group); wait until the GPU is free.
    if os.path.exists(f"{stem}.pid"):
        try:
            os.killpg(int(open(f"{stem}.pid").read()), signal.SIGTERM)
        except ProcessLookupError:
            pass
        os.remove(f"{stem}.pid")
    for _ in range(60):
        if gpu_used_mb(gpu) < 1000:
            return True
        time.sleep(2)
    print(f"GPU {gpu} is still busy after stopping {stem}")
    return False


def keep_log(stem, label):
    if os.path.exists(f"{stem}.log"):
        shutil.copy(f"{stem}.log", f"{PRIVATE}/{label}.log")


DONE, FAILED = [], []


def save(run_dir):
    # Aggregates to OUT, everything to PRIVATE, right after each run.
    name = os.path.basename(run_dir)
    shutil.copytree(run_dir, f"{PRIVATE}/runs/{name}", dirs_exist_ok=True)
    os.makedirs(f"{OUT}/runs/{name}", exist_ok=True)
    for file in ("config.yaml", "metrics.json"):
        if os.path.exists(f"{run_dir}/{file}"):
            shutil.copy(f"{run_dir}/{file}", f"{OUT}/runs/{name}/{file}")


def evaluate(config):
    before = set(glob.glob(f"runs/{config}-*"))
    code = run("eval.run", "--config", f"configs/{config}.yaml", "--allow-test-split")
    new = sorted(set(glob.glob(f"runs/{config}-*")) - before)
    for run_dir in new:
        save(run_dir)
    (DONE if code == 0 and new else FAILED).append(config)
    print(config, "done" if code == 0 else "FAILED")


for config in NO_GPU_RUNS:
    evaluate(config)
MAIN_READY = run("llm.serve", "--config", "configs/pipeline_test.yaml") == 0
FT_READY = run("llm.serve", "--config", "configs/pipeline_test.yaml", "--verifier") == 0
SMOKE_OK = MAIN_READY and run("llm.smoke", "--config", "configs/pipeline_test.yaml") == 0
print({"7B": MAIN_READY, "fine-tuned 3B": FT_READY, "smoke": SMOKE_OK})

## Runs with the fine-tuned verifier
Each run takes about 20-35 minutes for 64 cases; results are saved after each run.

In [ ]:
NEEDS_FT = {"pipeline_test", "pipeline_no_claim_splitting_test", "pipeline_no_code_execution_test",
            "pipeline_no_retrieval_test", "pipeline_no_escalation_test"}
for config in FT_RUNS:
    if not SMOKE_OK or (config in NEEDS_FT and not FT_READY):
        print("skipped (server not ready):", config)
        FAILED.append(config)
        continue
    evaluate(config)
keep_log("vllm_verifier", "vllm_ft3b")
stop_server("vllm_verifier", gpu=1)

## A4a: untuned 3B verifier

In [ ]:
if SMOKE_OK and run("llm.serve", "--config", "configs/pipeline_base_verifier_test.yaml",
                    "--verifier") == 0:
    for config in BASE_RUNS:
        evaluate(config)
else:
    print("skipped A4a (server not ready)")
    FAILED.extend(BASE_RUNS)
keep_log("vllm_verifier", "vllm_base3b")
stop_server("vllm_verifier", gpu=1)
print("done:", DONE)
print("failed or skipped:", FAILED)

## Results and statistics
Statistics (`eval/stats.py`, `docs/DECISIONS.md` section 15): paired permutation tests by
situation with Holm correction within each family, bootstrap 95% CIs.

In [ ]:
KEYS = ["exact_decision_accuracy", "wrongful_change_rate", "wrongful_refusal_rate",
        "escalation_precision", "escalation_recall", "tone_sensitivity", "pushback_flip_rate"]
RUN_DIRS = {}
for path in sorted(glob.glob("runs/*_test-*/metrics.json")):
    metrics = json.load(open(path))
    name = metrics["run_id"].rsplit("-", 3)[0]
    RUN_DIRS[name] = os.path.dirname(path)
    values = " ".join(f"{k}={metrics['overall'].get(k)}" for k in KEYS)
    print(f"{name:34s} errors={metrics['n_case_errors']} {values}")

FAMILIES = {  # family name -> (systems compared with pipeline_test, metrics)
    "rq1a_baselines": (["b0_reject_test", "b0_accept_test", "b1_test", "agent_test"],
                       ["wrongful_change_rate", "wrongful_refusal_rate"]),
    "rq1b_robustness": (["pipeline_no_claim_splitting_test", "pipeline_prompted_verifier_test",
                         "pipeline_base_verifier_test"],
                        ["tone_sensitivity", "pushback_flip_rate"]),
    "exploratory_accuracy": ([n for n in RUN_DIRS if n != "pipeline_test"],
                             ["exact_decision_accuracy", "wrongful_change_rate",
                              "wrongful_refusal_rate"]),
}
if "pipeline_test" in RUN_DIRS:
    for family, (others, metrics) in FAMILIES.items():
        dirs = [RUN_DIRS[o] for o in others if o in RUN_DIRS]
        if not dirs:
            continue
        done = subprocess.run(["python", "-m", "eval.stats", RUN_DIRS["pipeline_test"], *dirs,
                               "--metrics", *metrics, "--allow-test-split"],
                              capture_output=True, text=True)
        if done.returncode == 0:
            Path(f"{OUT}/stats_{family}.json").write_text(done.stdout)
            print(family, "written")
        else:
            print(family, "FAILED:", done.stderr[-2000:])

## Save outputs and stop the servers
Download both zips; only `test_metrics.zip` goes to the repo.

In [ ]:
keep_log("vllm", "vllm_7b")
json.dump({"done": DONE, "failed_or_skipped": FAILED, "verifier_merged_here": NEED_MERGE},
          open(f"{OUT}/run_summary.json", "w"), indent=2)
!pip freeze > {OUT}/pip_freeze.txt
!git rev-parse HEAD > {OUT}/commit.txt
shutil.make_archive("/kaggle/working/test_metrics", "zip", OUT)
shutil.make_archive("/kaggle/working/test_private", "zip", PRIVATE)
stop_server("vllm", gpu=0)
print("done")